# 02. 세션 마트 생성 및 검증

## 개요

- 목적: `docs/metrics.md`의 처리 방침을 집행해 세션 단위 마트 `mart_session`을 생성하고, raw(`events`)에서 독립 계산한 값과 대조 검증한다.
- 방침 원천: `docs/metrics.md`(단일 원천). 이 노트북은 방침을 변경하지 않는다.
- 분석 단위: 세션(마트 1행 = 유효 세션 1개).
- 검증: (a) 마트 행수 = raw 유효 세션 수, (b) 유형별 카운트 합 = raw price>=0 유형별 건수, (c) revenue 합 = raw purchase·price>0 합. 불일치 시 수정하지 않고 양쪽 수치를 남긴다.

In [1]:
import os
import re
import time
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{quote_plus(os.getenv('DB_PASSWORD'))}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

In [2]:
def find_sql(name):
    for base in (Path.cwd(), Path.cwd().parent):
        cand = base / 'sql' / name
        if cand.exists():
            return cand
    raise FileNotFoundError(f'sql/{name} 없음 (cwd={Path.cwd()})')

def load_queries(path):
    body = Path(path).read_text(encoding='utf-8')
    parts = re.split(r'(?m)^--\s*name:\s*(\w+).*$', body)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

Q = load_queries(find_sql('02_mart_session.sql'))

def run(name):
    return pd.read_sql(text(Q[name]), engine)

def execute(name):
    # -- 라인 주석 제거 후 세미콜론 분리 (주석 내 세미콜론 오분리 방지)
    body = re.sub(r'--[^\n]*', '', Q[name])
    with engine.begin() as conn:
        for stmt in [s for s in body.split(';') if s.strip()]:
            conn.execute(text(stmt))

## 1. 마트 생성

`docs/metrics.md` 방침 집행: user_session NOT NULL(WHERE), 단일 user_id·지속시간 ≤ 1일(HAVING), 카운트는 price<0 제외·price=0 포함, revenue는 purchase·price>0.

In [3]:
t0 = time.time()
execute('create_mart_session')
build_sec = time.time() - t0
print(f"mart_session 생성 완료 · {build_sec:.0f}s")

mart_session 생성 완료 · 191s


## 2. 검증 a — 마트 행수 = raw 유효 세션 수

In [4]:
mart_n = int(run('mart_rowcount').iloc[0, 0])
raw_n = int(run('raw_valid_count').iloc[0, 0])
print(f"마트 행수 = {mart_n:,}")
print(f"raw 유효 세션 = {raw_n:,}")
print(f"일치: {mart_n == raw_n}")

마트 행수 = 4,499,479
raw 유효 세션 = 4,499,479
일치: True


## 3. 검증 b — 유형별 카운트 합 = raw(price>=0, 유효 세션 범위)

In [5]:
mart_c = run('mart_type_counts').iloc[0]
raw_c = run('raw_type_counts').set_index('event_type')['건수']
order = ['view', 'cart', 'remove_from_cart', 'purchase']
cmp_b = pd.DataFrame({
    'mart': [int(mart_c[t]) for t in order],
    'raw': [int(raw_c[t]) for t in order],
}, index=order)
cmp_b['일치'] = cmp_b['mart'] == cmp_b['raw']
cmp_b

,mart,raw,일치
view,9284031,9284031,True
cart,5502430,5502430,True
remove_from_cart,3729756,3729756,True
purchase,1229578,1229578,True


## 4. 검증 c — revenue 합 = raw(purchase, price>0, 유효 세션 범위)

In [6]:
mart_r = float(run('mart_revenue').iloc[0, 0])
raw_r = float(run('raw_revenue').iloc[0, 0])
print(f"마트 revenue = {mart_r:,.2f}")
print(f"raw revenue = {raw_r:,.2f}")
print(f"일치: {round(mart_r, 2) == round(raw_r, 2)}")

마트 revenue = 6,088,823.52
raw revenue = 6,088,823.52
일치: True


## 5. 검증 종합

In [7]:
check_a = mart_n == raw_n
check_b = bool(cmp_b['일치'].all())
check_c = round(mart_r, 2) == round(raw_r, 2)
print(f"검증a(행수): {check_a}")
print(f"검증b(유형별 카운트): {check_b}")
print(f"검증c(revenue): {check_c}")
print(f"전체 통과: {check_a and check_b and check_c}")

검증a(행수): True
검증b(유형별 카운트): True
검증c(revenue): True
전체 통과: True
